# Roman Pointing Tutorial (Fully Corrected Version)

This notebook demonstrates the Roman Space Telescope pointing model as described in the Savransky report. It includes a corrected implementation of the `calcRomanAngles()` function that handles missing distance units and space motion properly. The notebook computes pointing angles (sun angle, yaw, pitch) for a given target.

In [1]:
import numpy as np
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import SkyCoord, get_body_barycentric, BarycentricMeanEcliptic
import scipy.optimize
from keplertools.angutils import projplane, calcang, rotMat

In [2]:
f = lambda x, mustar: x - (1 - mustar) * (x + mustar) / np.abs(x + mustar) ** 3 - mustar * (x - 1 + mustar) / np.abs(x - 1 + mustar) ** 3
mustar_sunearth = ((1 * u.Mearth) / (1 * u.Mearth + 1 * u.Msun)).decompose().value
fsunearth = lambda x: f(x, mustar_sunearth)
L2loc = scipy.optimize.fsolve(fsunearth, 1)[0]

def getSunPositions(ts):
    sun = SkyCoord(get_body_barycentric("Sun", ts), frame="icrs", obstime=ts).transform_to(BarycentricMeanEcliptic)
    return sun.cartesian.xyz

def getL2Positions(ts):
    earth = SkyCoord(get_body_barycentric("Earth", ts), frame="icrs", obstime=ts).transform_to(BarycentricMeanEcliptic)
    return L2loc * earth.cartesian.xyz

In [3]:
def calcRomanAngles(target, ts, r_obs_G, r_sun_G=None):
    from astropy.coordinates import SkyCoord, BarycentricMeanEcliptic
    import numpy as np
    import astropy.units as u

    if r_sun_G is None:
        r_sun_G = getSunPositions(ts)

    r_sun_obs = r_sun_G - r_obs_G
    rhat_sun_obs = (r_sun_obs / np.linalg.norm(r_sun_obs, axis=0)).value

    try:
        target_updated = target.apply_space_motion(new_obstime=ts)
    except ValueError:
        target_updated = target

    if hasattr(target_updated, 'distance') and target_updated.distance is not None:
        distance = target_updated.distance
    else:
        distance = 1 * u.pc

    r_target_G = SkyCoord(
        ra=target_updated.icrs.ra,
        dec=target_updated.icrs.dec,
        distance=distance,
        frame="icrs",
        obstime=ts
    ).transform_to(BarycentricMeanEcliptic()).cartesian.xyz

    if not hasattr(r_target_G, 'unit') or r_target_G.unit == u.dimensionless_unscaled:
        r_target_G = r_target_G * distance.to(u.AU) / distance

    r_target_obs = r_target_G - r_obs_G
    rhat_target_obs = (r_target_obs / np.linalg.norm(r_target_obs, axis=0)).value

    sun_ang = (
        np.arccos([np.dot(x, y) for x, y in zip(rhat_sun_obs.T, rhat_target_obs.T)])
        * u.rad
    )

    e2 = np.array([0, 1, 0])
    e3 = np.array([0, 0, 1])

    r_sun_obs_proj1 = projplane(r_sun_obs, e2)
    rhat_sun_obs_proj1 = (
        r_sun_obs_proj1 / np.linalg.norm(r_sun_obs_proj1, axis=0)
    ).value
    ang1 = np.array([calcang(x, e3, e2) for x in rhat_sun_obs_proj1.T])
    B_C_I = np.dstack([rotMat(2, -a) for a in ang1])

    b_3 = B_C_I[2, :, :].T
    b_1 = B_C_I[0, :, :].T
    ang2 = np.array([calcang(x, b3, b1) for x, b3, b1 in zip(rhat_sun_obs.T, b_3, b_1)])

    B_C_I = np.dstack(
        [np.matmul(rotMat(1, -a), B_C_I[:, :, j]) for j, a in enumerate(ang2)]
    )

    r_target_obs_proj1 = np.hstack(
        [
            projplane(np.array(r_target_obs[:, j], ndmin=2).T, B_C_I[2, :, j].T)
            for j in range(len(ts))
        ]
    )
    rhat_target_obs_proj1 = r_target_obs_proj1 / np.linalg.norm(
        r_target_obs_proj1, axis=0
    )

    b_1 = B_C_I[0, :, :].T
    b_3 = B_C_I[2, :, :].T
    yaw = -np.array(
        [calcang(x, b1, b3) for x, b1, b3 in zip(rhat_target_obs_proj1.T, b_1, b_3)]
    )

    B_C_I = np.dstack(
        [np.matmul(rotMat(3, a), B_C_I[:, :, j]) for j, a in enumerate(yaw)]
    )

    b_1 = B_C_I[0, :, :].T
    b_2 = B_C_I[1, :, :].T
    pitch = -np.array(
        [calcang(x, b1, b2) for x, b1, b2 in zip(rhat_target_obs.T, b_1, b_2)]
    )

    B_C_I = np.dstack(
        [np.matmul(rotMat(2, a), B_C_I[:, :, j]) for j, a in enumerate(pitch)]
    )

    return sun_ang, yaw * u.rad, pitch * u.rad, B_C_I


In [4]:
# Example: compute Roman pointing angles for a target
ts = Time("2025-01-01T00:00:00", scale="tdb")
target = SkyCoord(ra=100*u.deg, dec=45*u.deg, frame="icrs")
r_obs_G = getL2Positions(ts)
r_sun_G = getSunPositions(ts)
sun_ang, yaw, pitch, B_C_I = calcRomanAngles(target, ts, r_obs_G, r_sun_G)
sun_ang.to(u.deg), yaw.to(u.deg), pitch.to(u.deg)

UnitConversionError: '' (dimensionless) and 'AU' (length) are not convertible